# Steel Surface Defect Detection with U-Net

This notebook demonstrates training a U-Net model for steel surface defect segmentation
using the NEU Surface Defect Dataset.

## Overview
- **Dataset**: NEU Surface Defect Dataset (1800 grayscale images, 6 defect types)
- **Model**: U-Net (encoder-decoder with skip connections)
- **Task**: Multi-class segmentation of steel surface defects
- **Classes**: crazing, inclusion, patches, pitted_surface, rolled-in_scale, scratches

## 1. Setup and Dependencies

In [ ]:
import os
import sys
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms

# Add project root to path
sys.path.insert(0, '..')

from src.model import UNet
from src.dataset import NEUSegmentationDataset, NEU_CLASSES, get_train_transforms, get_val_transforms
from src.train import train_one_epoch, validate, dice_coefficient

print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 2. Download and Explore the Dataset

The NEU Surface Defect Dataset contains 1800 grayscale images (200x200 pixels) 
categorized into 6 types of surface defects commonly found in hot-rolled steel strips.

In [ ]:
# Download the NEU Surface Defect Dataset
# Source: http://faculty.neu.edu.cn/songkechen/en/zdylm/263265/list/index.htm
DATA_DIR = Path('../data/NEU-CLS')

if not DATA_DIR.exists():
    print('Downloading NEU Surface Defect Dataset...')
    os.makedirs('../data', exist_ok=True)
    
    # Using gdown to download from Google Drive (mirror)
    import gdown
    url = 'https://drive.google.com/uc?id=1qrdZlaDi272eA79b0uCwwqPrm2Q_WI3t'
    output = '../data/NEU-CLS.zip'
    gdown.download(url, output, quiet=False)
    
    # Extract
    with zipfile.ZipFile(output, 'r') as zip_ref:
        zip_ref.extractall('../data/')
    print('Dataset downloaded and extracted!')
else:
    print(f'Dataset already exists at {DATA_DIR}')

# Show dataset structure
if DATA_DIR.exists():
    for class_dir in sorted(DATA_DIR.iterdir()):
        if class_dir.is_dir():
            n_images = len(list(class_dir.glob('*')))
            print(f'  {class_dir.name}: {n_images} images')

In [ ]:
# Visualize sample images from each class
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
fig.suptitle('NEU Surface Defect Dataset - Sample Images', fontsize=14)

for idx, (ax, class_name) in enumerate(zip(axes.flat, NEU_CLASSES)):
    class_dir = DATA_DIR / class_name
    if class_dir.exists():
        images = sorted(class_dir.glob('*.bmp')) + sorted(class_dir.glob('*.jpg'))
        if images:
            from PIL import Image
            img = Image.open(images[0])
            ax.imshow(np.array(img), cmap='gray')
    ax.set_title(class_name)
    ax.axis('off')

plt.tight_layout()
os.makedirs('../assets', exist_ok=True)
plt.savefig('../assets/dataset_samples.png', dpi=100, bbox_inches='tight')
plt.show()

## 3. Create Data Loaders

In [ ]:
# Hyperparameters
IMAGE_SIZE = (200, 200)
BATCH_SIZE = 16
NUM_CLASSES = 7  # 6 defects + background
LEARNING_RATE = 1e-3
NUM_EPOCHS = 30

# Create datasets
train_dataset = NEUSegmentationDataset(
    root_dir=str(DATA_DIR),
    transform=get_train_transforms(IMAGE_SIZE),
    split='train',
    image_size=IMAGE_SIZE,
)

val_dataset = NEUSegmentationDataset(
    root_dir=str(DATA_DIR),
    transform=get_val_transforms(IMAGE_SIZE),
    split='val',
    image_size=IMAGE_SIZE,
)

print(f'Training samples: {len(train_dataset)}')
print(f'Validation samples: {len(val_dataset)}')

# Create data loaders
train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True
)

In [ ]:
# Visualize a batch with masks
images, masks = next(iter(train_loader))

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle('Training Batch - Images and Masks', fontsize=14)

for i in range(4):
    # Denormalize image
    img = images[i].numpy().transpose(1, 2, 0)
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    img = std * img + mean
    img = np.clip(img, 0, 1)
    
    axes[0, i].imshow(img)
    axes[0, i].set_title(f'Image {i}')
    axes[0, i].axis('off')
    
    axes[1, i].imshow(masks[i].numpy(), cmap='tab10', vmin=0, vmax=NUM_CLASSES-1)
    axes[1, i].set_title(f'Mask {i}')
    axes[1, i].axis('off')

plt.tight_layout()
plt.show()

## 4. Model Architecture

In [ ]:
# Initialize the U-Net model
model = UNet(in_channels=3, num_classes=NUM_CLASSES).to(device)

# Model summary
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'Total parameters: {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')
print(f'\nModel architecture:\n{model}')

In [ ]:
# Verify model output shape
dummy_input = torch.randn(1, 3, *IMAGE_SIZE).to(device)
dummy_output = model(dummy_input)
print(f'Input shape: {dummy_input.shape}')
print(f'Output shape: {dummy_output.shape}')
assert dummy_output.shape == (1, NUM_CLASSES, *IMAGE_SIZE), 'Output shape mismatch!'

## 5. Training

In [ ]:
# Loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=5, verbose=True
)

# Training loop
history = {'train_loss': [], 'val_loss': [], 'train_dice': [], 'val_dice': []}
best_val_dice = 0.0

os.makedirs('../models', exist_ok=True)

for epoch in range(NUM_EPOCHS):
    print(f'\nEpoch {epoch+1}/{NUM_EPOCHS}')
    print('-' * 40)
    
    # Train
    train_metrics = train_one_epoch(
        model, train_loader, optimizer, criterion, device, NUM_CLASSES
    )
    
    # Validate
    val_metrics = validate(model, val_loader, criterion, device, NUM_CLASSES)
    
    # Update scheduler
    scheduler.step(val_metrics['loss'])
    
    # Record history
    history['train_loss'].append(train_metrics['loss'])
    history['val_loss'].append(val_metrics['loss'])
    history['train_dice'].append(train_metrics['dice'])
    history['val_dice'].append(val_metrics['dice'])
    
    print(f"  Train Loss: {train_metrics['loss']:.4f} | Train Dice: {train_metrics['dice']:.4f}")
    print(f"  Val Loss:   {val_metrics['loss']:.4f} | Val Dice:   {val_metrics['dice']:.4f}")
    
    # Save best model
    if val_metrics['dice'] > best_val_dice:
        best_val_dice = val_metrics['dice']
        torch.save(model.state_dict(), '../models/unet_steel_defect.pth')
        print(f'  -> Saved best model (Dice: {best_val_dice:.4f})')

print(f'\nTraining complete! Best validation Dice: {best_val_dice:.4f}')

## 6. Training Results

In [ ]:
# Plot training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss
ax1.plot(history['train_loss'], label='Train Loss', linewidth=2)
ax1.plot(history['val_loss'], label='Val Loss', linewidth=2)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training and Validation Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Dice Score
ax2.plot(history['train_dice'], label='Train Dice', linewidth=2)
ax2.plot(history['val_dice'], label='Val Dice', linewidth=2)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Dice Coefficient')
ax2.set_title('Training and Validation Dice Score')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
os.makedirs('../assets', exist_ok=True)
plt.savefig('../assets/training_curves.png', dpi=100, bbox_inches='tight')
plt.show()

## 7. Inference and Visualization

In [ ]:
# Load best model
model.load_state_dict(torch.load('../models/unet_steel_defect.pth', map_location=device))
model.eval()

# Run inference on validation samples
fig, axes = plt.subplots(3, 3, figsize=(15, 15))
fig.suptitle('Inference Results: Input | Ground Truth | Prediction', fontsize=14)

val_iter = iter(val_loader)
images, masks = next(val_iter)

with torch.no_grad():
    outputs = model(images.to(device))
    preds = torch.argmax(outputs, dim=1).cpu()

for i in range(3):
    # Original image
    img = images[i].numpy().transpose(1, 2, 0)
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    img = std * img + mean
    img = np.clip(img, 0, 1)
    
    axes[i, 0].imshow(img)
    axes[i, 0].set_title('Input Image')
    axes[i, 0].axis('off')
    
    axes[i, 1].imshow(masks[i].numpy(), cmap='tab10', vmin=0, vmax=NUM_CLASSES-1)
    axes[i, 1].set_title('Ground Truth')
    axes[i, 1].axis('off')
    
    axes[i, 2].imshow(preds[i].numpy(), cmap='tab10', vmin=0, vmax=NUM_CLASSES-1)
    axes[i, 2].set_title('Prediction')
    axes[i, 2].axis('off')

plt.tight_layout()
plt.savefig('../assets/inference_results.png', dpi=100, bbox_inches='tight')
plt.show()

## 8. Export Model for Deployment

The trained model is saved and ready to be served via FastAPI.

In [ ]:
# Final model info
model_path = Path('../models/unet_steel_defect.pth')
if model_path.exists():
    model_size = model_path.stat().st_size / (1024 * 1024)
    print(f'Model saved at: {model_path}')
    print(f'Model size: {model_size:.2f} MB')
else:
    print('Model not yet trained. Run the training cells above.')

print('\n--- Deployment ---')
print('To serve the model with FastAPI:')
print('  uvicorn src.api:app --host 0.0.0.0 --port 7860')
print('\nTo build and run with Docker:')
print('  docker build -t steel-defect-detection .')
print('  docker run -p 7860:7860 steel-defect-detection')